# 📏 03 — Modelo Baseline: Logistic Regression

> **Credit Risk ML** · Notebook 3 de 5

**Objetivo:** estabelecer uma referência mínima de performance.  
Qualquer modelo mais complexo (RF, XGBoost) **precisa superar** esta baseline para justificar sua complexidade extra.

| Etapa | Descrição |
|-------|-----------|
| 3.1 | Por que Logistic Regression como baseline? |
| 3.2 | Treinamento |
| 3.3 | Avaliação completa |
| 3.4 | Curva ROC |
| 3.5 | Matriz de confusão |
| 3.6 | Coeficientes (interpretabilidade) |
| 3.7 | Análise de erros |

---

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import joblib
from sklearn.linear_model  import LogisticRegression
from sklearn.metrics import (
    roc_auc_score, roc_curve, auc, f1_score, precision_score, recall_score,
    accuracy_score, confusion_matrix, ConfusionMatrixDisplay, classification_report,
    precision_recall_curve,
)
from pathlib import Path

plt.rcParams.update({
    'figure.facecolor':'#0F1117','axes.facecolor':'#161B22','axes.edgecolor':'#30363D',
    'axes.labelcolor':'#E6EDF3','xtick.color':'#8B949E','ytick.color':'#8B949E',
    'text.color':'#E6EDF3','grid.color':'#21262D','grid.linewidth':0.8,
    'figure.dpi':120,'font.family':'monospace',
})
C = {'lr':'#58A6FF','good':'#3FB950','bad':'#F78166',
     'accent':'#D2A8FF','warn':'#E3B341','muted':'#8B949E'}
SEED = 42
print('✅  Imports OK')

In [ ]:
# Carrega artefatos do notebook 02
splits_path = Path('../models/splits.pkl')

if splits_path.exists():
    splits = joblib.load(splits_path)
    X_train_sc = splits['X_train_sc']
    X_test_sc  = splits['X_test_sc']
    y_train    = splits['y_train']
    y_test     = splits['y_test']
    features   = splits['features']
    print('✅  Dados carregados do notebook 02')
else:
    print('⚠️  Execute o notebook 02 primeiro para gerar os artefatos.')
    print('   Ou rode o bloco abaixo para recriar os dados inline.')

print(f'   Treino: {len(X_train_sc):,} | Teste: {len(X_test_sc):,} | Features: {len(features)}')

---
### 3.1 Por que Logistic Regression?

```
VANTAGENS como baseline de crédito:
  ✅ Rápida de treinar e validar
  ✅ Coeficientes têm interpretação direta (log-odds)
  ✅ Probabilidades naturalmente calibradas
  ✅ Regulada por C (L2 por padrão) — evita overfitting
  ✅ Exigida em alguns contextos regulatórios (Basel, IFRS 9)

LIMITAÇÕES:
  ❌ Assume linearidade entre features e log-odds
  ❌ Sensível a multicolinearidade
  ❌ Não captura interações entre features automaticamente
```

---
### 3.2 Treinamento

In [ ]:
import time

def train_baseline(X_train, y_train, C=1.0, save_path=None):
    """
    Treina Logistic Regression com configurações recomendadas para crédito.

    Parâmetros chave:
      C             : regularização inversa (menor = mais regularizado)
      class_weight  : 'balanced' compensa desbalanceamento automaticamente
      max_iter=1000 : garante convergência mesmo com features correlacionadas
    """
    model = LogisticRegression(
        C            = C,
        max_iter     = 1_000,
        class_weight = 'balanced',
        solver       = 'lbfgs',
        random_state = SEED,
        n_jobs       = -1,
    )
    t0 = time.time()
    model.fit(X_train, y_train)
    elapsed = time.time() - t0
    if save_path:
        joblib.dump(model, save_path)
    return model, elapsed

lr_model, lr_time = train_baseline(
    X_train_sc, y_train, save_path='../models/logistic_regression.pkl'
)
print(f'✅  Modelo treinado em {lr_time:.2f}s')
print(f'   Iterações convergência: {lr_model.n_iter_[0]}')

---
### 3.3 Avaliação Completa

In [ ]:
def evaluate_model(model, X_test, y_test, name='Model', threshold=0.5):
    """Calcula e exibe métricas completas de classificação."""
    y_prob = model.predict_proba(X_test)[:, 1]
    y_pred = (y_prob >= threshold).astype(int)

    metrics = {
        'name'     : name,
        'auc'      : roc_auc_score(y_test, y_prob),
        'accuracy' : accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred, zero_division=0),
        'recall'   : recall_score(y_test, y_pred, zero_division=0),
        'f1'       : f1_score(y_test, y_pred, zero_division=0),
    }

    print(f'\n{'─'*52}')
    print(f'  {name}')
    print(f'{'─'*52}')
    print(f'  ROC-AUC   : {metrics["auc"]:.4f}')
    print(f'  Accuracy  : {metrics["accuracy"]:.4f}')
    print(f'  Precision : {metrics["precision"]:.4f}   (dos aprovados, quantos eram bons?)')
    print(f'  Recall    : {metrics["recall"]:.4f}   (dos defaults, quantos capturamos?)')
    print(f'  F1-Score  : {metrics["f1"]:.4f}')
    print(f'\n{classification_report(y_test, y_pred, target_names=["Bom","Default"])}')
    return metrics, y_prob, y_pred

lr_metrics, lr_prob, lr_pred = evaluate_model(lr_model, X_test_sc, y_test, 'Logistic Regression')

---
### 3.4 Curva ROC + Precision-Recall

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.patch.set_facecolor('#0F1117')

# ROC
ax1 = axes[0]
fpr, tpr, thresholds_roc = roc_curve(y_test, lr_prob)
roc_auc_val = auc(fpr, tpr)
ax1.plot(fpr, tpr, lw=2.5, color=C['lr'], label=f'LR (AUC={roc_auc_val:.3f})')
ax1.fill_between(fpr, tpr, alpha=0.08, color=C['lr'])
ax1.plot([0,1],[0,1],'k--',lw=1,alpha=0.5,label='Random (AUC=0.50)')
ax1.set_xlabel('False Positive Rate (1 - Especificidade)', fontsize=11)
ax1.set_ylabel('True Positive Rate (Sensibilidade)', fontsize=11)
ax1.set_title('Curva ROC — Logistic Regression',
              fontsize=13, fontweight='bold', color='#E6EDF3')
ax1.legend(fontsize=10, framealpha=0.3); ax1.grid(alpha=0.2)

# Precision-Recall (mais informativa para classes desbalanceadas)
ax2 = axes[1]
prec, rec, _ = precision_recall_curve(y_test, lr_prob)
pr_auc = auc(rec, prec)
ax2.plot(rec, prec, lw=2.5, color=C['lr'], label=f'LR (PR-AUC={pr_auc:.3f})')
ax2.fill_between(rec, prec, alpha=0.08, color=C['lr'])
baseline_pr = y_test.mean()
ax2.axhline(baseline_pr, ls='--', color='gray', lw=1,
            label=f'Random ({baseline_pr:.2f})')
ax2.set_xlabel('Recall', fontsize=11)
ax2.set_ylabel('Precision', fontsize=11)
ax2.set_title('Curva Precision-Recall',
              fontsize=13, fontweight='bold', color='#E6EDF3')
ax2.legend(fontsize=10, framealpha=0.3); ax2.grid(alpha=0.2)

plt.tight_layout(); plt.show()

---
### 3.5 Matriz de Confusão

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.patch.set_facecolor('#0F1117')

for ax, (normalize, title) in zip(axes, [
    (None,   'Contagem Absoluta'),
    ('true', 'Normalizada por Linha'),
]):
    cm = confusion_matrix(y_test, lr_pred, normalize=normalize)
    disp = ConfusionMatrixDisplay(cm, display_labels=['Bom','Default'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(title, fontsize=12, color='#E6EDF3', pad=10)
    if normalize:
        for text in ax.texts:
            text.set_text(f'{float(text.get_text()):.1%}')

plt.suptitle('Matriz de Confusão — Logistic Regression',
             fontsize=14, color=C['accent'], fontweight='bold')
plt.tight_layout(); plt.show()

tn, fp, fn, tp = confusion_matrix(y_test, lr_pred).ravel()
print(f'  Verdadeiros Negativos (Bom aprovado)  : {tn:,}')
print(f'  Falsos Positivos    (Bom rejeitado)   : {fp:,}')
print(f'  Falsos Negativos    (Ruim aprovado)   : {fn:,}  ← custo alto!')
print(f'  Verdadeiros Positivos (Ruim rejeitado): {tp:,}')

---
### 3.6 Coeficientes — Interpretabilidade

In [ ]:
coefs = pd.Series(lr_model.coef_[0], index=features).sort_values()

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.patch.set_facecolor('#0F1117')

# Coeficientes brutos
ax1 = axes[0]
colors_bar = [C['bad'] if v > 0 else C['good'] for v in coefs.values]
ax1.barh(coefs.index, coefs.values, color=colors_bar, edgecolor='#0F1117', alpha=0.9)
ax1.axvline(0, color='#8B949E', lw=1)
ax1.set_title('Coeficientes (positivo=↑ risco)', fontsize=12, color='#E6EDF3')
ax1.set_xlabel('Valor do coeficiente', fontsize=11)
ax1.grid(axis='x', alpha=0.3)

# Odds Ratio (mais intuitivo de explicar)
ax2 = axes[1]
odds_ratio = np.exp(coefs).sort_values()
colors_or  = [C['bad'] if v > 1 else C['good'] for v in odds_ratio.values]
ax2.barh(odds_ratio.index, odds_ratio.values, color=colors_or, edgecolor='#0F1117', alpha=0.9)
ax2.axvline(1, color='#8B949E', lw=1.5, ls='--')
ax2.set_title('Odds Ratio (>1 = ↑ risco, <1 = ↓ risco)', fontsize=12, color='#E6EDF3')
ax2.set_xlabel('exp(coeficiente)', fontsize=11)
ax2.grid(axis='x', alpha=0.3)

plt.suptitle('Interpretabilidade — Logistic Regression',
             fontsize=14, color=C['accent'], fontweight='bold')
plt.tight_layout(); plt.show()

print('\n  Interpretação dos top 3 coeficientes positivos:')
top3 = coefs.nlargest(3)
for feat, val in top3.items():
    or_val = np.exp(val)
    print(f'  {feat:<22} coef={val:.3f}  OR={or_val:.2f}x  '
          f'(+1σ nesta feature → {or_val:.1f}× mais chance de default)')

---
### 3.7 Análise de Erros

In [ ]:
# Identifica os 4 quadrantes de erro
df_errors = X_test_sc.copy()
df_errors['y_true']  = y_test.values
df_errors['y_prob']  = lr_prob
df_errors['y_pred']  = lr_pred
df_errors['quadrant'] = (
    df_errors.apply(lambda r:
        'TP' if r.y_true==1 and r.y_pred==1 else
        'TN' if r.y_true==0 and r.y_pred==0 else
        'FP' if r.y_true==0 and r.y_pred==1 else 'FN',
        axis=1)
)

# Distribuição de scores por quadrante (onde o modelo tem mais incerteza?)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.patch.set_facecolor('#0F1117')

ax1 = axes[0]
for quad, color in [('TN',C['good']),('TP','#58A6FF'),('FP',C['warn']),('FN',C['bad'])]:
    subset = df_errors[df_errors['quadrant']==quad]['y_prob']
    if len(subset): ax1.hist(subset, bins=40, alpha=0.6, color=color,
                             label=f'{quad} (n={len(subset):,})', density=True)
ax1.axvline(0.5, color='white', lw=1, ls='--', label='threshold=0.5')
ax1.set_title('Distribuição de Scores por Quadrante', fontsize=12, color='#E6EDF3')
ax1.set_xlabel('Score de Default'); ax1.legend(fontsize=9, framealpha=0.2); ax1.grid(alpha=0.2)

# Clientes com maior erro: FN (defaults aprovados) — mais custosos
ax2 = axes[1]
fn_clients = df_errors[df_errors['quadrant']=='FN']
fn_scores  = fn_clients['y_prob'].values
ax2.hist(fn_scores, bins=40, color=C['bad'], alpha=0.8, edgecolor='#0F1117')
ax2.axvline(fn_scores.mean(), color=C['warn'], lw=2, ls='--',
            label=f'Média={fn_scores.mean():.2f}')
ax2.set_title(f'Scores dos Falsos Negativos (n={len(fn_clients):,})\n'
              f'Defaults que o modelo aprovou — maior custo',
              fontsize=11, color='#E6EDF3')
ax2.set_xlabel('Score de Default'); ax2.legend(fontsize=9); ax2.grid(alpha=0.2)

plt.tight_layout(); plt.show()

print('\n  Análise de erros:')
print(f'  FN (defaults aprovados) : {(df_errors.quadrant=="FN").sum():,}  → prejuízo máximo')
print(f'  FP (bons rejeitados)    : {(df_errors.quadrant=="FP").sum():,}  → custo de oportunidade')
print(f'\n  → Notebooks 04/05 mostrarão modelos que reduzem os FN')

# Salva métricas baseline para comparação
joblib.dump(lr_metrics, '../models/lr_metrics.pkl')
print('\n✅  Baseline salvo. Próximo: 04_model_advanced.ipynb')